In [0]:
# ============================================================
# INSTALL PACKAGE
# ============================================================

%pip install yfinance


# ============================================================
# YFINANCE INGESTION PIPELINE
# Raw Data Collector
# Stores data in Unity Catalog Volume
# ============================================================

import yfinance as yf
import json
import time
import random
import pandas as pd

from concurrent.futures import ThreadPoolExecutor, as_completed
from pyspark.sql import functions as F


# ------------------------------------------------------------
# BASE PATH
# ------------------------------------------------------------

BASE_PATH = "/Volumes/company_risk_intelligence_platform/bronze/raw_data/yfinance/"


# ------------------------------------------------------------
# CURRENT DATE USING PYSPARK
# ------------------------------------------------------------

today = (
    spark.range(1)
    .select(
        F.date_format(
            F.current_date(),
            "yyyy/MM/dd"
        ).alias("date")
    )
    .collect()[0]["date"]
)

# ------------------------------------------------------------
# DATASET FOLDERS
# ------------------------------------------------------------

FOLDERS = [
    "stock",
    "news",
    "income_statement",
    "balance_sheet",
    "cashflow",
    "info"
]


# ------------------------------------------------------------
# BUILD DYNAMIC PATH
# ------------------------------------------------------------

def build_path(folder):
    return f"{BASE_PATH}{folder}/{today}/"


# ------------------------------------------------------------
# Company List 
# ------------------------------------------------------------

companies = [
    {"name": "J SAINSBURY PLC", "ticker": "SBRY.L"},
    {"name": "JD SPORTS FASHION PLC", "ticker": "JD.L"},
    {"name": "OCADO GROUP PLC", "ticker": "OCDO.L"},
    {"name": "LLOYDS BANKING GROUP PLC", "ticker": "LLOY.L"},
    {"name": "NATWEST GROUP PLC", "ticker": "NWG.L"},
    {"name": "NATIONAL GRID PLC", "ticker": "NG.L"},
    {"name": "SSE PLC", "ticker": "SSE.L"},
    {"name": "DRAX GROUP PLC", "ticker": "DRX.L"},
    {"name": "BALFOUR BEATTY PLC", "ticker": "BBY.L"},
    {"name": "PERSIMMON PLC", "ticker": "PSN.L"},
    {"name": "EASYJET PLC", "ticker": "EZJ.L"},
    {"name": "INTERCONTINENTAL HOTELS GROUP PLC", "ticker": "IHG.L"},
    {"name": "HIKMA PHARMACEUTICALS PLC", "ticker": "HIK.L"},
    {"name": "PEARSON PLC", "ticker": "PSON.L"},
    {"name": "BT GROUP PLC", "ticker": "BT-A.L"},
    {"name": "ITV PLC", "ticker": "ITV.L"},
    {"name": "JOHNSON MATTHEY PLC", "ticker": "JMAT.L"},
    {"name": "MELROSE INDUSTRIES PLC", "ticker": "MRO.L"},
    {"name": "AVIVA PLC", "ticker": "AV.L"},
    {"name": "ASTON MARTIN LAGONDA GLOBAL HOLDINGS PLC", "ticker": "AML.L"},
]


# ------------------------------------------------------------
# SAFE DATAFRAME CONVERTER
# Fixes Timestamp + NaN issues
# ------------------------------------------------------------

def safe_df_to_records(df):

    if df is None or df.empty:
        return []

    df = df.copy()

    # convert timestamp columns -> string
    df.columns = [str(c) for c in df.columns]

    # reset index
    df = df.reset_index()

    # replace NaN -> None
    df = df.where(pd.notnull(df), None)

    return df.to_dict(orient="records")


# ------------------------------------------------------------
# SAFE JSON WRITER
# ------------------------------------------------------------

def save_json(folder, filename, data):

    path = f"{build_path(folder)}{filename}"

    if data is None:
        data = []

    json_data = json.dumps(
        data,
        indent=2,
        default=str
    )

    (
        spark
        .createDataFrame([(json_data,)], ["json_data"])
        .coalesce(1)
        .write
        .mode("overwrite")
        .text(path)
    )


# ------------------------------------------------------------
# INGEST SINGLE COMPANY
# ------------------------------------------------------------

def ingest_company(company):

    ticker = company["ticker"]
    name = company["name"]

    # normalize ticker for filename safety
    safe_ticker = (
        ticker
        .replace(".", "_")
        .replace("-", "_")
    )

    print(f"Processing {name} ({ticker})")

    try:

        stock = yf.Ticker(ticker)

        # ---------------- STOCK ----------------

        try:

            hist = stock.history(period="1y")

            save_json(
                "stock",
                f"{safe_ticker}",
                safe_df_to_records(hist)
            )

            print(f"stock success -> {ticker}")

        except Exception as e:

            print(f"stock failed -> {ticker}: {e}")

        # ---------------- NEWS ----------------

        try:

            save_json(
                "news",
                f"{safe_ticker}",
                stock.news
            )

            print(f"news success -> {ticker}")

        except Exception as e:

            print(f"news failed -> {ticker}: {e}")

        # ---------------- INCOME STATEMENT ----------------

        try:

            income = stock.financials

            save_json(
                "income_statement",
                f"{safe_ticker}",
                safe_df_to_records(income)
            )

            print(f"income_statement success -> {ticker}")

        except Exception as e:

            print(f"income_statement failed -> {ticker}: {e}")

        # ---------------- BALANCE SHEET ----------------

        try:

            bs = stock.balance_sheet

            save_json(
                "balance_sheet",
                f"{safe_ticker}",
                safe_df_to_records(bs)
            )

            print(f"balance_sheet success -> {ticker}")

        except Exception as e:

            print(f"balance_sheet failed -> {ticker}: {e}")

        # ---------------- CASHFLOW ----------------

        try:

            cf = stock.cashflow

            save_json(
                "cashflow",
                f"{safe_ticker}",
                safe_df_to_records(cf)
            )

            print(f"cashflow success -> {ticker}")

        except Exception as e:

            print(f"cashflow failed -> {ticker}: {e}")

        # ---------------- INFO ----------------

        try:

            save_json(
                "info",
                f"{safe_ticker}",
                stock.info
            )

            print(f"info success -> {ticker}")

        except Exception as e:

            print(f"info failed -> {ticker}: {e}")

    except Exception as e:

        print(f"FAILED -> {ticker}: {e}")

    # avoid Yahoo rate limiting
    time.sleep(1 + random.random())


# ------------------------------------------------------------
# PARALLEL EXECUTION
# ------------------------------------------------------------

def run_pipeline():

    with ThreadPoolExecutor(max_workers=4) as executor:

        futures = [
            executor.submit(ingest_company, company)
            for company in companies
        ]

        for future in as_completed(futures):

            try:
                future.result()

            except Exception as e:
                print("Worker error:", e)


# ------------------------------------------------------------
# RUN INGESTION
# ------------------------------------------------------------

run_pipeline()

print("YFinance Ingestion Completed")


In [0]:
# -------------------------------------------------
# INGESTION HEALTH CHECK
# -------------------------------------------------

expected = set([
    c["ticker"]
    .replace(".", "_")
    .replace("-", "_")
    for c in companies
])

print("\nINGESTION CHECK\n")

for folder in FOLDERS:

    path = build_path(folder)

    try:
        files = dbutils.fs.ls(path)

        # -------------------------------------------------
        # extract filename WITHOUT extension dependency
        # -------------------------------------------------
        existing = set([
            f.name.split(".")[0].replace("/", "")
            for f in files
            if f.name.strip() != ""
        ])

        missing = expected - existing

        print("=" * 50)
        print("DATASET:", folder)
        print("FILES:", len(existing))
        print("MISSING:", len(missing))

        if missing:
            print("MISSING FILES:", list(missing))

    except Exception as e:

        print("=" * 50)
        print("DATASET:", folder)
        print("STATUS: PATH NOT FOUND")
        print("ERROR:", e)